In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
len(docs)

3

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splitter_docs = splitter.split_documents(docs)

In [6]:
len(splitter_docs)

5

In [8]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Mahesh Yadav\Desktop\Gen-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1769.33it/s]


In [9]:
vector_store = InMemoryVectorStore.from_documents(
    documents=splitter_docs,
    embedding=embeddings
)

In [13]:
# same_record = vector_store.similarity_search("patient name")
# same_record[0].page_content

#### agent = tools, llm ,prompt

In [35]:
@tool
def retriever_tool(query:str):
   """
      This tool can help you to retrive the relevant date of the PDF Document, and these documents have details about medical reports.


   """
   print("Tool Called:",query)
   docs = vector_store.similarity_search(query=query, k =4)
   context = ""

   for doc in docs:
      context = doc.page_content + "\n\n"
      
   return context

In [36]:
llm =ChatGroq(model="openai/gpt-oss-20b")

In [37]:
System_Prompt = """
    You are a helpfull assistent that answer questions using retrieved context.
    ALWAYS use the 'retriever_tool' tool for quations requiring external knowledge
"""

In [38]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [45]:
query = "What is the name of patient, and What is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called: patient name and doctors names medical report
Tool Called: patient name in medical report "Physician's Signature: Dr."
Tool Called: "Physician's Signature" "Dr. Alan Green" patient name


In [46]:
result = response["messages"][-1].content

In [47]:
print(result)

**Patient name:** Ms. Johnson  
**Doctor(s) name:** Dr. Alan Green, MD
